## ⚙️ Data Harmonization Pipeline: FDA vs. RASFF
**Paper:** *The Schizophrenia of Global Food Safety: A Comparative Longitudinal Analysis (2008--2025)* **Author:** Md. Shafin Ahamed  

### 📝 Overview
This notebook acts as the **ETL (Extract, Transform, Load) engine** for the study. It ingests raw regulatory enforcement data from the US FDA and EU RASFF systems and processes them into a functionally equivalent, harmonized dataset.

### 🔧 Key Processing Steps
1.  **Data Ingestion:** Loads previously cleaned `FDA Import Refusals` and `RASFF Portal Notifications`.
2.  **Scope Filtration:**
    * **FDA:** Filters for "Human Food" only (Industry Codes `02`-`42`) and removes Administrative/Feed records.
    * **RASFF:** Filters for "Border Rejections" only (excluding Alerts/Information) to match FDA "Refusals."
    * **Temporal:** Truncates both datasets to the overlapping window of **2008--2025**.
3.  **Taxonomy Harmonization:**
    * **Trade Sectors:** Maps 100+ FDA Product Codes and EU Product Categories into **4 Harmonized Trade Sectors** (e.g., Seafood, Nuts & Seeds, Fruit/Veg).
    * **Hazard Classes:** Implements a "Many-to-One" dictionary to map thousands of violation charges into **7 Harmonized Hazard Categories**.
    * *Crucial Logic:* Separates **"Natural Toxins"** (Mycotoxins) from general **"Chemicals"** (Pesticides/Vet Drugs).

### 📂 Inputs & Outputs
* **Input:** `food_refusals_complete.csv` (FDA Data), `RASFF_Alert_Notifications_Cleaned.csv` (RASFF Data).
* **Output:** `FDA_Import_Refusals_Human_Foods_harmonized.csv`, `RASFF_Alert_Notifications_harmonized.csv`.

In [2]:
import pandas as pd
import numpy as np

# --------------------
# Load Data
# --------------------
df_fda = pd.read_csv(
    r"D:\data_files\Journal\DataSet_of_FDA_RASFF\PreCleaned_data\food_refusals_complete.csv",
    low_memory=False
)

df_rasff = pd.read_csv(
    r"D:\data_files\Journal\DataSet_of_FDA_RASFF\PreCleaned_data\RASFF_Alert_Notifications_(1979-2025)_Cleaned.csv",
    low_memory=False
)


print("Done!")


Done!


In [21]:
# =========================
# RASFF ↔ FDA HARMONIZATION
# =========================

# -------- Alert Mapping (RASFF) --------
alert_map = {
    'border rejection': 'Border Rejection',
    'alert': 'Alert',
    'information': 'Information',
    'information for attention': 'Attention',
    'border rejection notification': 'Border Rejection',
    'information for follow-up': 'Follow-up',
    'alert notification': 'Alert',
    'information notification for attention': 'Attention',
    'information notification for follow-up': 'Follow-up',
    'non-compliance notification': 'Others'
}

df_rasff['Alert_Classification_clean'] = (
    df_rasff['Alert_Classification']
    .str.lower()
    .str.strip()
    .map(alert_map)
)

df_rasff_border = df_rasff[df_rasff['Alert_Classification_clean'] == 'Border Rejection' ]

# -------- FDA Sector Mapping --------
fda_map = {
    'Fishery/Seafood': 'Seafood',
    'Vegetables': 'Agriculture',
    'Fruit/Juices': 'Agriculture',
    'Veg Products': 'Agriculture',
    'Fruit': 'Agriculture',
    'Nuts/Seeds': 'Nuts & Seeds',
    'Spices': 'Spices',
    'Spices/Flavors': 'Spices'
}

df_fda['Sector'] = df_fda['Product_category'].map(fda_map)
df_fda_clean = df_fda.dropna(subset=['Sector']).copy()

# -------- RASFF Sector Mapping --------
rasff_map = {
    # Seafood Group (Merging 5 categories)
    'fish and fish products': 'Seafood',
    'crustaceans and products thereof': 'Seafood',
    'cephalopods and products thereof': 'Seafood',
    'bivalve molluscs and products thereof': 'Seafood',
    'gastropods': 'Seafood',
    
    # Agriculture Group
    'fruits and vegetables': 'Agriculture',
    
    # Nuts Group
    'nuts, nut products and seeds': 'Nuts & Seeds',
    
    # Spices Group
    'herbs and spices': 'Spices'
}

df_rasff_border['Sector'] = (
    df_rasff_border['Product_category']
    .str.lower()
    .str.strip()
    .map(rasff_map)
)

df_rasff_clean = df_rasff_border.dropna(subset=['Sector']).copy()

# -------- Final Verification --------
print("\n--- Final RASFF Sector Counts ---")
print(df_rasff_clean['Sector'].value_counts())
print(f"Total RASFF Rows Kept: {len(df_rasff_clean)}")

print("\n--- FDA Sector Counts ---")
print(df_fda_clean['Sector'].value_counts())
print(f"Total FDA Rows Kept: {len(df_fda_clean)}")



--- Final RASFF Sector Counts ---
Sector
Nuts & Seeds    7510
Agriculture     7375
Seafood         3382
Spices          2230
Name: count, dtype: int64
Total RASFF Rows Kept: 20497

--- FDA Sector Counts ---
Sector
Agriculture     58452
Seafood         40987
Spices          15305
Nuts & Seeds     1825
Name: count, dtype: int64
Total FDA Rows Kept: 116569


C:\Users\DELL\AppData\Local\Temp\ipykernel_20660\2697709436.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rasff_border['Sector'] = (


In [22]:
def map_fda_to_rasff(reason_text):
    """
    Parses FDA Violation Charges and maps them to RASFF Hazard Groups.
    Prioritizes high-risk categories if multiple charges exist.
    """
    if pd.isna(reason_text):
        return "Other"
    
    text = str(reason_text).upper()

    # 1. BIOLOGICAL HAZARDS (Highest Priority) 
    # Keywords: Salmonella, Listeria, Bacteria, Vibrio, E. Coli
    if any(x in text for x in ['SALMONELLA', 'LISTERIA', 'BACTERIA', 'VIBRIO', 'E. COLI', 'SHIGELLA', 'HEPATITIS']):
        return "Biological"

    # 2. NATURAL TOXINS 
    # Keywords: Aflatoxin, Mycotoxin, Histamine
    if any(x in text for x in ['AFLATOXIN', 'MYCOTOXIN', 'HISTAMINE', 'PATULIN']):
        return "Natural Toxins"

    # 3. ALLERGENS & ADDITIVES 
    # Keywords: Allergen, Sulfites, Yellow #5, Color additives
    if any(x in text for x in ['ALLERGEN', 'SULFITE', 'YELLOW #5', 'COLOR ADD', 'DYE']):
        return "Allergens & Additives"

    # 4. CHEMICAL HAZARDS 
    # Keywords: Pesticide, Unsafe Color, Unsafe Additive, Drug Residues, Lead, Poison
    if any(x in text for x in ['PESTICIDE', 'UNSAFE COL', 'UNSAFE ADD', 'VETDRUG', 'NITROFURAN', 'CHLORAMP', 'POISON', 'LEAD', 'MERCURY', 'PB-FOOD', 'MELAMINE']):
        return "Chemical"

    # 5. PHYSICAL & PROCESS HAZARDS 
    # Keywords: Filthy (Insect/Rodent), Foreign Object, Insanitary, Glass, Metal
    # FDA "Filthy" usually implies extraneous matter or decomposition.
    if any(x in text for x in ['FILTHY', 'FOREIGN OB', 'IMBED OBJT', 'INSANITARY', 'MFR INSAN', 'CHOKE', 'GLASS', 'METAL', 'PLASTIC']):
        return "Physical & Process"
    
    # Process issues (Canning defects, leaks)
    if any(x in text for x in ['NEEDS FCE', 'NO PROCESS', 'ACID', 'LEAK', 'SWELL', 'LACF']):
        # Note: "No Process" often refers to administrative filing for canned foods, 
        # but it represents a process safety risk.
        return "Physical & Process"

    # 6. REGULATORY & QUALITY 
    # Keywords: Labeling, English, Standard of Identity, Registration
    if any(x in text for x in ['LABEL', 'LBL', 'ENGLISH', 'INGRE', 'STD IDENT', 'MISBRAND', 'FSVP', 'REGIST', 'ORDER']):
        return "Regulatory & Quality"

    # 7. OTHER 
    return "Other"

# Apply the mapping
print("Mapping FDA reasons to RASFF Standards...")
df_fda_clean['Hazard_Group'] = df_fda_clean['Reason'].apply(map_fda_to_rasff)

# Check the results
print("\nDistribution of FDA Risks (RASFF Standardized) ")
print(df_fda_clean['Hazard_Group'].value_counts())

Mapping FDA reasons to RASFF Standards...

Distribution of FDA Risks (RASFF Standardized) 
Hazard_Group
Physical & Process       44284
Chemical                 32289
Biological               22303
Regulatory & Quality      9740
Other                     5159
Natural Toxins            1594
Allergens & Additives     1200
Name: count, dtype: int64


In [23]:
#Year Harmonization for FDA, 2008 to 2025
# According to analysis: RASFF's Border Rejection alert started in 2008.
df_fda_clean = df_fda_clean[
(df_fda_clean['Year'] >=2008) & (df_fda_clean['Year'] <=2025)
]
print('Done!')

Done!


In [24]:
# Save both files for Visualizations
df_fda_clean.to_csv(r"D:\data_files\Journal\data\FDA_Import_Refusals_Human_Foods_harmonized.csv", index=False)
df_rasff_clean.to_csv(r"D:\data_files\Journal\data\RASFF_Alert_Notifications_(1979-2025)_harmonized.csv", index=False)
print('Saved!')

Saved!
